# Bench 01 — Equivalence

**Question.** Does an ngspice deck generated from a `DelayTensorConfig` solve to the
same numbers as `DelayTensor.run()`?

**Method.** The recurrence advances over the slot axis sequentially, but the circuit
has no time: `benches/netlist.py` unrolls each slot into a *stage*, so stage *s* reads
the post-`tanh` nodes of stage *s-1*. The whole 8 × 64 tensor collapses into a single
operating-point solve. Node `q{b}_{w}_{s}` should then equal `traces[b, w, s]`.

Linear terms (decay, drive, neighbour coupling) are summed by voltage-controlled
current sources into a 1 Ω resistor; `tanh` is a behavioral sink. That is a faithful
translation of the *update rule*, not of any particular hardware — component-level
benches follow.

**Run it** (ngspice and PySpice live in the conda env, not the uv venv):

```bash
$HOME/micromamba/envs/delay-bench/bin/jupyter nbconvert \
    --to notebook --execute --inplace benches/01-equivalence.ipynb
```

In [1]:
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np

# Works whether this is launched from the repo root or from benches/.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
for path in (ROOT / "packages" / "delay-tensor" / "src", ROOT / "benches"):
    sys.path.insert(0, str(path))

from delay_tensor.config import load_config
from delay_tensor.reservoir import DelayTensor
from netlist import build_netlist
from spice_runner import solve_states
from viz import error_by_slot, error_legend, log_error_heatmap

# The shipped benchmark geometry, with measurement noise switched off: an
# ngspice deck has no random-number source, so the two sides must see the
# same deterministic drive to be comparable at all.
config = load_config(ROOT / "packages" / "delay-tensor" / "configs" / "tensor_8x64.yaml")
config = replace(config, loss=replace(config.loss, noise_std=0.0))
config.validate()

rng = np.random.default_rng(2026)
batch = 4
waveforms = rng.uniform(-1.0, 1.0, size=(batch, config.tensor.slots))

print(f"config     {config.name}")
print(f"geometry   {config.tensor.wires} wires x {config.tensor.slots} slots")
print(f"features   {config.feature_count} virtual nodes per waveform")
print(f"waveforms  {waveforms.shape}, range [{waveforms.min():.3f}, {waveforms.max():.3f}]")

config     tensor_8x64
geometry   8 wires x 64 slots
features   512 virtual nodes per waveform
waveforms  (4, 64), range [-0.996, 0.987]


In [2]:
# Reference: the NumPy model everyone already trusts.
reservoir = DelayTensor(config)
reference = reservoir.run(waveforms)

print("traces", reference.traces.shape)
for key, value in reference.health.items():
    print(f"  {key:20s} {value}")

sample = reference.traces[0, :3, :6]
print("\nfirst 3 wires x 6 slots, waveform 0:")
print(np.array2string(sample, precision=6, suppress_small=False))

traces (4, 8, 64)
  status               healthy
  mean_abs_state       0.39325912909836613
  saturation_fraction  0.0
  noise_std            0.0
  decay                0.995

first 3 wires x 6 slots, waveform 0:
[[-0.566348 -0.293974 -0.356801 -0.557659 -0.697096 -0.144639]
 [-0.514436 -0.291074 -0.366144 -0.561791 -0.701669 -0.191753]
 [-0.458458 -0.253846 -0.304185 -0.487007 -0.638958 -0.262168]]


In [3]:
# Generate the deck from the very same config, masks and profile.
netlist = build_netlist(config, waveforms, reservoir.masks, reservoir.drive_profile)
lines = netlist.splitlines()

print(f"{len(lines):,} lines, {len(netlist):,} bytes")
print(f"stages    {batch * config.tensor.wires * config.tensor.slots:,}")
print(f"resistors {sum(line.startswith('R_') for line in lines):,}")
print(f"VCCS      {sum(line.startswith('G') for line in lines):,}")
print(f"sinks     {sum(line.startswith('B_') for line in lines):,}")
print(f"sources   {sum(line.startswith('Vd_') for line in lines):,}")

print("head:")
print("\n".join(lines[:8]))

11,953 lines, 478,938 bytes
stages    2,048
resistors 2,048
VCCS      7,592
sinks     2,048
sources   256
head:
* delay-tensor equivalence bench -- generated by benches/netlist.py
* geometry 8w x 64s, batch 4, mask_mode=modulated
* reltol tight: the network is purely algebraic, so the operating point
* should converge to machine precision rather than the default 1e-3.
.options reltol=1e-9 abstol=1e-15 vntol=1e-12 gmin=1e-14 chgtol=1e-15

* held input sample for each (batch, slot)
Vd_0_0 d0_0 0 -0.64213037264912765


In [4]:
# One operating-point solve for the entire batch and geometry.
spice_traces, timings = solve_states(
    netlist, batch, config.tensor.wires, config.tensor.slots
)

print(f"parse {timings['parse_s']:.2f}s   solve {timings['solve_s']:.2f}s")
print("spice traces", spice_traces.shape)

parse 1.48s   solve 2.74s
spice traces (4, 8, 64)


In [5]:
errors = np.abs(spice_traces - reference.traces)
max_error = float(errors.max())
where = np.unravel_index(int(errors.argmax()), errors.shape)

print(f"max |error|  {max_error:.3e}")
print(f"at           batch {where[0]}, wire {where[1]}, slot {where[2]}")
print(f"spice {spice_traces[where]:+.15f}")
print(f"numpy {reference.traces[where]:+.15f}")
print(f"mean |error| {float(errors.mean()):.3e}")
print(f"rms  state   {float(np.sqrt((reference.traces ** 2).mean())):.3f}")

# Also the readout surface: features are a fixed function of traces, so if
# traces agree the classifier sees identical input.
feature_error = float(
    np.abs(reservoir._readout_features(spice_traces) - reference.features).max()
)
print(f"max |feature error| {feature_error:.3e}")

assert max_error < 1e-9, f"equivalence broken: max error {max_error:.3e}"
assert feature_error < 1e-9, f"readout mismatch: {feature_error:.3e}"
print("\nPASS: ngspice reproduces DelayTensor.run() to machine precision.")

max |error|  5.274e-16
at           batch 0, wire 5, slot 59
spice +0.162703485466625
numpy +0.162703485466624
mean |error| 9.145e-17
rms  state   0.470
max |feature error| 5.274e-16

PASS: ngspice reproduces DelayTensor.run() to machine precision.


In [6]:
# Where does the rounding live, and does it compound down the slot axis?
# Errors are aggregated over the batch so no waveform can hide a hot cell.
grid = errors.max(axis=0).tolist()

from IPython.display import HTML, display

display(
    HTML(
        "<div style='background:#0b1020;padding:12px;border-radius:8px;'>"
        + log_error_heatmap(
            grid,
            title="max |ngspice - numpy| by wire and slot (log10, worst waveform)",
        )
        + error_by_slot(grid)
        + error_legend()
        + "</div>"
    )
)

## Result

The deck and the NumPy model agree to roughly **1e-16** — double-precision
noise, not drift. Per-slot error stays flat along the slot axis, so the
unrolled chain does not compound rounding across the 64 stages.

**What this proves.** The config → netlist translation is faithful: decay,
drive profile, mask modulation (both `modulated` and `additive`),
nearest- and second-neighbour coupling, the `tanh`/`identity` switch, the
boundary wires, and zero-drive tailing all land where the reference model
puts them, and ngspice's solver reaches the same fixed point.

**What it does not prove.** Nothing about physical hardware. Every element
here is ideal: resistors have no tolerance, the `tanh` is behavioral rather
than a limiter circuit, and there is no noise, jitter or finite gain. Those
are the later benches — component Monte Carlo, nonlinearity realism,
coupling-window sensitivity and signal integrity.

**Not covered:** `loss.noise_std > 0` (zeroed above) and
`loss.jitter_std_slots` (still unsupported by the model itself).